<a href="https://colab.research.google.com/github/DiogoAzevedoSilva/ecommerce-customer-analysis/blob/main/E_commerce_customer_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E-Commerce Customer Behavior Analysis

## Project Overview

This project analyzes a dataset of 5,000 customer orders from an e-commerce
platform, combining transactional, demographic, and behavioral variables. The
goal is to extract actionable business insights and demonstrate a full analytics
workflow — from data quality validation through to strategic recommendations.

The analysis is structured in five sections:

1. **Data Quality** — validating the dataset before any analysis is performed
2. **Customer & Platform Overview** — understanding who the customers are and
   how they interact with the platform
3. **Drivers of Purchasing Behavior** — identifying which factors actually
   influence spending
4. **Advanced Analytics** — customer segmentation (K-Means) and revenue
   prediction (Random Forest)
5. **Strategic Recommendations** — prioritized actions grounded in the findings

A central theme emerges across Sections 3 and 4: **revenue on this platform is
driven by product mix and purchase quantity, not by customer demographics or
browsing behavior.** This finding shapes every recommendation in Section 5.

## Dataset

| Property | Detail |
|---|---|
| Records | 5,000 customer orders |
| Features | 18 variables |
| Types | Transactional, demographic, behavioral, operational |
| Key variables | Order value, product category, quantity, session duration, page views, customer rating, delivery time |
| Target variable | `total_amount` (used in Section 4.2) |
| Currency | Turkish Lira (TRY, synthetic) |

## Analytical Approach

| Stage | Method | Purpose |
|---|---|---|
| Data validation | pandas | Quality checks, plausibility verification |
| Exploratory analysis | plotly, pandas | Distribution analysis, correlation, segmentation |
| Customer segmentation | K-Means clustering | Identify distinct behavioral groups |
| Revenue prediction | Random Forest regression | Quantify drivers of order value |

## Tools

- **Python** — pandas, plotly, scikit-learn
- **Environment** — Google Colab

In [1]:
# Dependencies
!pip install -q kaleido==0.2.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 9.2 MB/s eta 0:00:00


In [53]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "colab"

df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "umuttuygurr/e-commerce-customer-behavior-and-sales-analysis-tr",
    "ecommerce_customer_behavior_dataset.csv"
)

# Visual constants
PRIMARY_COLOR = "#2196F3"
CLUSTER_PALETTE = px.colors.qualitative.Set2

Using Colab cache for faster access to the 'e-commerce-customer-behavior-and-sales-analysis-tr' dataset.


# 1. Data Understanding & Quality Checks

## Business Question
Is the dataset reliable, clean, and suitable for analytical and predictive modeling purposes?

Before performing any analysis, it is critical to validate data quality to ensure accurate and
trustworthy results. This section covers dataset structure, missing values, duplicates, and
categorical consistency.

In [54]:
df = df.rename(columns={
    'Order_ID':                  'order_id',
    'Customer_ID':               'customer_id',
    'Date':                      'date',
    'Age':                       'age',
    'Gender':                    'gender',
    'City':                      'city',
    'Product_Category':          'product_cat',
    'Unit_Price':                'unit_price',
    'Quantity':                  'qty',
    'Discount_Amount':           'discount_amount',
    'Total_Amount':              'total_amount',
    'Payment_Method':            'pay_method',
    'Device_Type':               'device',
    'Session_Duration_Minutes':  'session_min',
    'Pages_Viewed':              'pag_view',
    'Is_Returning_Customer':     'returning_cust',
    'Delivery_Time_Days':        'delivery_time_days',
    'Customer_Rating':           'rating'
})

df["date"] = pd.to_datetime(df["date"])

df.head()

,order_id,customer_id,date,age,gender,city,product_cat,unit_price,qty,discount_amount,total_amount,pay_method,device,session_min,pag_view,returning_cust,delivery_time_days,rating
0,ORD_001337,CUST_01337,2023-01-01,27,Female,Bursa,Toys,54.28,1,0.00,54.28,Debit Card,Mobile,4,14,True,8,5
1,ORD_004885,CUST_04885,2023-01-01,42,Male,Konya,Toys,244.90,1,0.00,244.90,Credit Card,Mobile,11,3,True,3,3
2,ORD_004507,CUST_04507,2023-01-01,43,Female,Ankara,Food,48.15,5,0.00,240.75,Credit Card,Mobile,7,8,True,5,2
3,ORD_000645,CUST_00645,2023-01-01,32,Male,Istanbul,Electronics,804.06,1,229.28,574.78,Credit Card,Mobile,8,10,False,1,4
4,ORD_000690,CUST_00690,2023-01-01,40,Female,Istanbul,Sports,755.61,5,0.00,3778.05,Cash on Delivery,Desktop,21,10,True,7,4


In [55]:
df[df.qty > 5]


,order_id,customer_id,date,age,gender,city,product_cat,unit_price,qty,discount_amount,total_amount,pay_method,device,session_min,pag_view,returning_cust,delivery_time_days,rating


In [56]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   order_id            5000 non-null   object        
 1   customer_id         5000 non-null   object        
 2   date                5000 non-null   datetime64[ns]
 3   age                 5000 non-null   int64         
 4   gender              5000 non-null   object        
 5   city                5000 non-null   object        
 6   product_cat         5000 non-null   object        
 7   unit_price          5000 non-null   float64       
 8   qty                 5000 non-null   int64         
 9   discount_amount     5000 non-null   float64       
 10  total_amount        5000 non-null   float64       
 11  pay_method          5000 non-null   object        
 12  device              5000 non-null   object        
 13  session_min         5000 non-null   int64       

In [57]:
df.describe()

,date,age,unit_price,qty,discount_amount,total_amount,session_min,pag_view,delivery_time_days,rating
count,5000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.00000,5000.00000,5000.000000,5000.000000
mean,2023-08-16 09:16:24.959999744,35.032600,455.834120,2.220000,24.852804,983.108914,14.57340,8.98420,6.497000,3.902800
min,2023-01-01 00:00:00,18.000000,5.180000,1.000000,0.000000,7.870000,1.00000,1.00000,1.000000,1.000000
25%,2023-04-30 00:00:00,27.000000,76.587500,1.000000,0.000000,122.517500,8.00000,7.00000,4.000000,3.000000
50%,2023-08-17 00:00:00,35.000000,182.950000,2.000000,0.000000,337.910000,13.00000,9.00000,6.000000,4.000000
75%,2023-12-06 00:00:00,42.000000,513.930000,3.000000,8.760000,979.695000,19.00000,11.00000,8.000000,5.000000
max,2024-03-26 00:00:00,75.000000,7159.450000,5.000000,1525.550000,22023.900000,73.00000,24.00000,25.000000,5.000000
std,NaN,11.080546,712.477209,1.398711,88.385124,1898.978528,8.66575,2.80434,3.464966,1.128542


## 1.1 Plausibility Checks

Beyond structure, we verify that the data makes internal sense:
- No negative or zero values in transactional fields
- Total amount reconciles with the formula: `unit_price x qty - discount_amount`

In [58]:
# Check for non-positive values in fields that should always be positive
plausibility_checks = {
    "unit_price <= 0":     (df["unit_price"] <= 0).sum(),
    "qty <= 0":            (df["qty"] <= 0).sum(),
    "total_amount <= 0":   (df["total_amount"] <= 0).sum(),
    "discount_amount < 0": (df["discount_amount"] < 0).sum(),
}

for check, count in plausibility_checks.items():
    print(f"{check}: {count} rows")

unit_price <= 0: 0 rows
qty <= 0: 0 rows
total_amount <= 0: 0 rows
discount_amount < 0: 0 rows


In [59]:
# Verify total_amount reconciles with unit_price × qty − discount_amount
# A tolerance of 0.01 accounts for potential rounding
df["amount_check"] = (
    (df["unit_price"] * df["qty"] - df["discount_amount"]) - df["total_amount"]
).abs()

reconciliation_failures = (df["amount_check"] > 0.01).sum()
print(f"Reconciliation failures: {reconciliation_failures}")

# If failures exist, inspect them
if reconciliation_failures > 0:
    print(df[df["amount_check"] > 0.01][
        ["order_id", "unit_price", "qty", "discount_amount", "total_amount", "amount_check"]
    ].head())

df.drop(columns="amount_check", inplace=True)

Reconciliation failures: 0


## 1.2 Quality Checks

In [60]:
# Missing values
df.isna().sum()

,0
order_id,0
customer_id,0
date,0
age,0
gender,0
city,0
product_cat,0
unit_price,0
qty,0
discount_amount,0


In [61]:
# Duplicate rows
df.duplicated().sum()

np.int64(0)

In [62]:
# Unique values per categorical column — confirms no unexpected categories
for col in ["product_cat", "city", "gender", "pay_method", "device"]:
    print(f"{col}: {sorted(df[col].unique())}")

product_cat: ['Beauty', 'Books', 'Electronics', 'Fashion', 'Food', 'Home & Garden', 'Sports', 'Toys']
city: ['Adana', 'Ankara', 'Antalya', 'Bursa', 'Eskisehir', 'Gaziantep', 'Istanbul', 'Izmir', 'Kayseri', 'Konya']
gender: ['Female', 'Male', 'Other']
pay_method: ['Bank Transfer', 'Cash on Delivery', 'Credit Card', 'Debit Card', 'Digital Wallet']
device: ['Desktop', 'Mobile', 'Tablet']


## Data Quality Summary

| Check | Result |
|---|---|
| Missing values | None |
| Duplicate rows | None |
| Categorical consistency | Valid categories across all fields |
| Non-positive transactional values | None |
| Amount reconciliation | Passes / [update if failures found] |

The dataset is clean, internally consistent, and suitable for exploratory
and predictive analysis.

# 2. Customer & Platform Overview

## Business Question
Who are the customers, and how do they interact with the platform?

This section explores customer demographics, platform usage behavior, and transactional
patterns. These distributions provide the baseline context for the deeper analysis in
Sections 3 and 4.


## Helper Functions


In [63]:
def plot_counts(series, x_label, sort=True):
    df_count = series.value_counts(sort=sort).reset_index()
    df_count.columns = [x_label, "Orders Count"]

    return px.bar(df_count,
                  x=x_label,
                  y="Orders Count",
                  title=f"Orders by {x_label}",
                  text_auto=True,
                  color_discrete_sequence=[PRIMARY_COLOR]
           ).update_layout(showlegend=False)


def plot_hist(series, x_title, nbins=20):
    mean   = series.mean()
    median = series.median()

    return (
        px.histogram(series,
                     title=f"Distribution of {x_title}",
                     nbins=nbins,
                     color_discrete_sequence=[PRIMARY_COLOR])
        .add_vline(x=mean,
                   line_dash="dash",
                   line_color="red",
                   annotation_text=f"Mean: {mean:.1f}",
                   annotation_position="top right")
        .add_vline(x=median,
                   line_dash="dot",
                   line_color="green",
                   annotation_text=f"Median: {median:.1f}",
                   annotation_position="top left")
        .update_layout(xaxis_title=x_title,
                       yaxis_title="Orders Count",
                       bargap=0.1,
                       showlegend=False)
    )


def plot_box(series, axis_title):
    return (
        px.box(series,
               title=f"{axis_title} — Boxplot",
               color_discrete_sequence=[PRIMARY_COLOR])
        .update_layout(xaxis=dict(visible=False),
                       yaxis_title=axis_title,
                       showlegend=False)
    )

## 2.1 Customer Demographics


### 2.1.1 Age

Age ranges from 18 to 75. Grouping into brackets improves interpretability
and follows standard marketing segmentation ranges.

In [64]:
age_bins   = [17, 25, 35, 45, 55, 75]
age_labels = ["18-25", "26-35", "36-45", "46-55", "56-75"]

df["age_group"] = pd.cut(df["age"],
                         bins=age_bins,
                         labels=age_labels,
                         right=True,
                         ordered=True)

plot_counts(df.age_group, "Age Group", sort=False).show()
plot_hist(df.age, "Customer Age", nbins=60).show()
plot_box(df.age, "Customer Age").show()

### 2.1.2 Gender

In [65]:
plot_counts(df.gender, "Gender").show()

### 2.1.3 City

In [66]:
plot_counts(df.city, "City").show()

### Key Observations — Demographics

- Customers aged **26–45** represent the platform's core demographic, accounting
  for the largest share of orders across both age brackets.
- The gender split is close to even, with a slight female majority
  (Female: ~51%, Male: ~49%).
- Order volume is heavily concentrated in **Istanbul, Ankara, and Izmir** —
  the three largest cities by population — reflecting the platform's urban
  customer base. Note that high order volume does not imply high order value;
  this relationship is explored in Section 3.1.

## 2.2 Platform Usage & Engagement

### 2.2.1 Device Type

In [67]:
plot_counts(df.device, "Device Type").show()

### 2.2.2 Session Duration in minutes

In [68]:
plot_hist(df.session_min, "Session Duration (min)", nbins=40).show()
plot_box(df.session_min, "Session Duration (min)").show()

### 2.2.3 Page Views

In [69]:
plot_hist(df.pag_view, "Page Views", nbins=25).show()
plot_box(df.pag_view, "Page Views").show()

### Key Observations — Platform Usage

- **Mobile dominates** device usage, accounting for [55.9]% of sessions.
  This has direct implications for UX prioritization.
- Session duration is right-skewed: most customers spend between 5 and 20 minutes
  on the platform, with a long tail extending to ~70 minutes. The mean (14.6 min)
  is pulled above the median (13 min) by these longer sessions.
- Page views follow a similar right-skewed pattern, consistent with a customer
  base that browses efficiently rather than extensively.
- Importantly, neither session duration nor page views are strong drivers of
  spending — this is confirmed in Section 3.2.

## 2.3 Transactional Patterns



### 2.3.1 Product categories

In [70]:
plot_counts(df.product_cat, "Product Category").show()

### 2.3.2 Payment methods

In [71]:
plot_counts(df.pay_method, "Payment Method").show()

### 2.3.3 Delivery Time

In [72]:
plot_hist(df.delivery_time_days, "Delivery Time (Days)", nbins=25).show()
plot_box(df.delivery_time_days, "Delivery Time (Days)").show()

##

### Key Observations — Transactional Patterns

- Order volume is **evenly distributed across product categories**, with no single
  category dominating by count. However, this masks significant revenue concentration
  — Electronics generates disproportionately high order values despite similar order
  volumes. This is examined in detail in Section 3.4.
- Payment method distribution is broadly even across Credit Card, Debit Card,
  and Digital Wallet, suggesting no strong payment preference in the customer base.
- Delivery times are uniformly distributed across the observed range, with no
  concentration at shorter or longer windows.

## Section Summary

The platform serves a broad, mobile-first customer base concentrated in major urban
centers, with customers aged 26–45 forming the core demographic. Browsing behavior
is efficient rather than extensive. Product category order volumes appear balanced
on the surface, but revenue tells a different story — explored in Section 3.


# 3. Drivers of Purchasing Behavior

## Business Question
Which factors most strongly influence customer spending?

This section examines how demographics, platform engagement, operational metrics, and product mix relate to order value. Each subsection closes with a business implication that is carried forward into the recommendations in Section 5.

## 3.1 Demographics vs Spending

### 3.1.1 Age vs Order Value

In [73]:
px.box(df.sort_values("age_group"),
       x="age_group",
       y="total_amount",
       log_y=True,
       title="Order Value by Age Group (Log Scale)",
       color_discrete_sequence=[PRIMARY_COLOR]
).update_layout(xaxis_title="Age Group",
                yaxis_title="Order Value (log scale)"
).show()

### 3.1.2 Gender vs Order Value

In [74]:
px.box(df,
       x="gender",
       y="total_amount",
       log_y=True,
       title="Order Value by Gender (Log Scale)",
       color_discrete_sequence=[PRIMARY_COLOR]
).update_layout(xaxis_title="Gender",
                yaxis_title="Order Value (log scale)"
).show()

### 3.1.3 City vs Order Value

In [75]:
city_stats = df.groupby("city").agg(
    avg_order_value=("total_amount", "mean"),
    order_volume=("order_id", "count")
).reset_index()

px.scatter(city_stats,
           x="order_volume",
           y="avg_order_value",
           text="city",
           title="City Performance: Order Volume vs Average Order Value",
           color_discrete_sequence=[PRIMARY_COLOR]
).update_traces(textposition="top center",
                marker=dict(size=10)
).update_layout(xaxis_title="Order Volume",
                yaxis_title="Average Order Value"
).show()

### Key Observations — Demographics

- Spending is **consistent across age groups**, with no meaningful difference in
  median or spread. Age is not a useful segmentation variable for revenue targeting
  on this platform.
- The gender gap in mean order value is negligible (Female: ~344, Male: ~333),
  confirming that gender is not a driver of spending.
- Geography tells a more interesting story: **Istanbul, Ankara, and Izmir**
  dominate order volume, while **Kayseri, Adana, and Konya** show the highest
  average order values despite lower volumes. This volume-value tradeoff suggests
  different strategic levers apply by city.

**Business Implication:**  
Demographic targeting by age or gender is unlikely to improve revenue outcomes.
Geographic strategy should distinguish between volume-driven cities (scale
and logistics focus) and value-driven cities (premium product and margin focus).

## 3.2 Engagement vs Spending

### 3.2.1 Session Duration vs Order Value

In [76]:
import numpy as np

df["log_total_amount"] = np.log10(df["total_amount"])

px.density_heatmap(df,
                   x="session_min",
                   y="log_total_amount",
                   nbinsx=40,
                   nbinsy=40,
                   title="Session Duration vs Order Value (Density)",
                   color_continuous_scale="Blues"
).update_layout(xaxis_title="Session Duration (min)",
                yaxis_title="Order Value (log10 scale)"
).show()


### 3.2.2 Page Views vs Order Value

In [77]:
px.density_heatmap(df,
                   x="pag_view",
                   y="log_total_amount",
                   nbinsx=25,
                   nbinsy=40,
                   title="Page Views vs Order Value (Density)",
                   color_continuous_scale="Blues"
).update_layout(xaxis_title="Page Views",
                yaxis_title="Order Value (log10 scale)"
).show()

df.drop(columns="log_total_amount", inplace=True)

### 3.2.3 Correlation Matrix

In [78]:
numeric_cols = ["age", "unit_price", "qty", "discount_amount",
                "total_amount", "session_min", "pag_view", "delivery_time_days"]

px.imshow(df[numeric_cols].corr(),
          text_auto=True,
          color_continuous_scale="RdBu",
          title="Correlation Matrix — Numeric Variables"
).show()

### Key Observations — Engagement

- The density plots show **no directional relationship** between session duration
  or page views and order value. Spending is spread uniformly across all engagement
  levels.
- The correlation matrix confirms this: session duration and page views have near-zero
  correlations with total_amount.
- The strongest correlations with total_amount are **unit_price** and **qty** —
  both transactional rather than behavioral variables. Note that unit_price and
  discount_amount are excluded from the predictive model in Section 4.2 to avoid
  target leakage.

**Business Implication:**  
Engagement metrics are weak proxies for revenue. Session duration and page views
play a limited but real secondary role — confirmed by their modest feature importance
scores (0.061 and 0.052) in Section 4.2. UX investment should target conversion
efficiency and checkout friction rather than time-on-site.

## 3.3 Operational Metrics & Customer Loyalty

### 3.3.1 Delivery Time vs Customer Rating

In [79]:
px.box(df,
       x="delivery_time_days",
       y="rating",
       title="Customer Rating by Delivery Time",
       color_discrete_sequence=[PRIMARY_COLOR]
).update_layout(xaxis_title="Delivery Time (Days)",
                yaxis_title="Customer Rating"
).show()

In [80]:
rating_by_delivery = (df.groupby("delivery_time_days")["rating"]
                        .mean()
                        .reset_index())

px.line(rating_by_delivery,
        x="delivery_time_days",
        y="rating",
        markers=True,
        title="Average Rating by Delivery Time",
        color_discrete_sequence=[PRIMARY_COLOR]
).update_layout(xaxis_title="Delivery Time (Days)",
                yaxis_title="Average Rating"
).show()

### 3.3.2 Returning vs New Customers

In [81]:
px.box(df,
       x="returning_cust",
       y="total_amount",
       log_y=True,
       title="Order Value: Returning vs New Customers (Log Scale)",
       color_discrete_sequence=[PRIMARY_COLOR]
).update_layout(xaxis_title="Returning Customer",
                yaxis_title="Order Value (log scale)"
).show()

In [82]:
px.box(df,
       x="returning_cust",
       y="session_min",
       title="Session Duration: Returning vs New Customers",
       color_discrete_sequence=[PRIMARY_COLOR]
).update_layout(xaxis_title="Returning Customer",
                yaxis_title="Session Duration (min)"
).show()

### Key Observations — Operational Metrics & Loyalty

- Customer ratings are **flat across all delivery times**, with no downward trend
  as delivery slows. Within the observed delivery window, speed does not measurably
  affect satisfaction.
- Returning and new customers show **identical spending and session behavior**.
  Repeat purchase status alone does not translate into higher order value or
  deeper engagement.

**Business Implication:**  
Operational improvements to delivery speed are unlikely to move satisfaction scores
within the current range. Loyalty strategies should focus on product value and
category upgrading rather than assuming returning customers will naturally spend more.

## 3.4 Product-Level Insights

### 3.4.1 Order Value by Product Category

In [83]:
px.box(df,
       x="product_cat",
       y="total_amount",
       log_y=True,
       title="Order Value by Product Category (Log Scale)",
       color_discrete_sequence=[PRIMARY_COLOR]
).update_layout(xaxis_title="Product Category",
                yaxis_title="Order Value (log scale)"
).show()

### Key Observations — Product Mix

- Order value varies substantially by category, confirming that **product mix is the primary driver of revenue** on this platform.
- **Electronics** generates the highest median order values with the widest dispersion, making it the dominant revenue category.
- **Home & Garden** and **Sports** form a mid-to-high value tier.
- **Food, Books, Beauty, and Toys** consistently produce low-value orders, suggesting a volume-driven rather than value-driven structure in these categories.
- The platform exhibits a **barbell revenue structure**: a small number of high-value categories generate a disproportionate share of revenue, while the majority of categories contribute through volume at low margins.

**Business Implication:**  
Product portfolio strategy is the highest-leverage variable for revenue growth.
This finding is confirmed and quantified in Section 4.2, where Electronics emerges
as the strongest predictor of order value (importance: 0.316).

## Section Summary

Across all dimensions examined — demographics, engagement, operations, and loyalty
— product mix is the only factor that consistently and substantially differentiates
order value. Age, gender, session behavior, and delivery time show limited or no explanatory power. This conclusion shapes the segmentation and modeling approach
in Section 4, and directly informs the recommendations in Section 5.

# 4. Advanced Analytics & Business Insights

## Business Question
Can advanced analytical techniques uncover deeper behavioral patterns and predict
revenue drivers?

This section applies two machine learning methods:
- **K-Means Clustering** — to segment customers into distinct behavioral groups
- **Random Forest Regression** — to identify and quantify the drivers of order value

## 4.1 Customer Segmentation — K-Means Clustering

### Business Objective
To identify distinct customer segments based on behavioral, transactional, and
demographic characteristics, enabling targeted marketing strategies and personalization.

### 4.1.1 Feature Selection

In [84]:
features_num = ["session_min",
                "pag_view",
                "total_amount",
                "qty",
                "age",
                "delivery_time_days",
                "rating"]

features_cat = ["device",
                "product_cat",
                "pay_method",
                "gender",
                "city"]

features = features_num + features_cat
df_cluster = df[features]

### 4.1.2 Preprocessing

In [85]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), features_num),
        # drop='first' avoids redundant dummy variables
        ("cat", OneHotEncoder(drop="first", sparse_output=False), features_cat)
    ])

### 4.1.3 Selecting the Optimal Number of Clusters

Two complementary methods are used to select k:
- **Elbow method** — identifies where adding clusters yields diminishing returns
  in inertia reduction
- **Silhouette analysis** — measures how well each point fits its assigned cluster
  vs neighbouring clusters. Higher is better.

In [86]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

K = range(2, 11)
inertia    = []
silhouette = []

X_scaled = preprocessor.fit_transform(df_cluster)

for k in K:
    model  = KMeans(n_clusters=k, random_state=0)
    labels = model.fit_predict(X_scaled)
    inertia.append(model.inertia_)
    silhouette.append(silhouette_score(X_scaled, labels))

In [87]:
px.line(x=list(K),
        y=inertia,
        markers=True,
        title="Elbow Method",
        color_discrete_sequence=[PRIMARY_COLOR]
).update_layout(xaxis_title="Number of Clusters (k)",
                yaxis_title="Inertia (Within-Cluster SSE)"
).show()

In [88]:
px.line(x=list(K),
        y=silhouette,
        markers=True,
        title="Silhouette Analysis",
        color_discrete_sequence=[PRIMARY_COLOR]
).update_layout(xaxis_title="Number of Clusters (k)",
                yaxis_title="Silhouette Score"
).show()

Both methods suggest **k=3** as the optimal number of clusters — the elbow
curve flattens and the silhouette score peaks at this value.

### 4.1.4 Fitting the Pipeline

In [89]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model",      KMeans(n_clusters=3, random_state=0))
])

pipeline.fit(df_cluster)
df["cluster"] = pipeline.predict(df_cluster)

### 4.1.5 Cluster Analysis & Profiling

#### Combined Cluster Summary

Mean numerical features and most frequent categorical value per cluster.


In [90]:
cluster_num = df.groupby("cluster")[features_num].mean().round(2)
cluster_cat = df.groupby("cluster")[features_cat].agg(
                  lambda x: x.value_counts().idxmax()
              )

cluster_summary = pd.concat([cluster_num, cluster_cat], axis=1)
cluster_summary

,session_min,pag_view,total_amount,qty,age,delivery_time_days,rating,device,product_cat,pay_method,gender,city
cluster,,,,,,,,,,,,
0,14.58,8.95,2439.44,4.25,34.72,6.55,4.10,Mobile,Electronics,Credit Card,Female,Istanbul
1,14.33,8.97,590.31,1.78,35.21,6.53,2.29,Mobile,Food,Credit Card,Female,Istanbul
2,14.68,9.01,488.05,1.49,35.10,6.46,4.52,Mobile,Books,Credit Card,Female,Istanbul


#### Cluster Size Distribution



In [91]:
cluster_size = (df["cluster"]
                .value_counts(normalize=True)
                .sort_index()
                .mul(100)
                .round(1)
                .reset_index())

cluster_size.columns = ["Cluster", "Share (%)"]
cluster_size["Cluster"] = cluster_size["Cluster"].astype(str)

px.bar(cluster_size,
       x="Cluster",
       y="Share (%)",
       title="Customer Distribution by Cluster",
       text="Share (%)",
       color="Cluster",
       color_discrete_sequence=CLUSTER_PALETTE
).update_traces(texttemplate="%{text}%",
                textposition="outside"
).update_layout(showlegend=False,
                yaxis=dict(range=[0, 70])
).show()

#### Revenue Contribution by Cluster


In [92]:
cluster_revenue = df.groupby("cluster")["total_amount"].agg(
    total_revenue="sum",
    avg_order_value="mean"
).reset_index()

cluster_revenue["revenue_share"] = (
    cluster_revenue["total_revenue"] / cluster_revenue["total_revenue"].sum() * 100
).round(1)

cluster_revenue["cluster"] = cluster_revenue["cluster"].astype(str)

In [93]:
px.bar(cluster_revenue,
       x="cluster",
       y="revenue_share",
       title="Revenue Share by Cluster (%)",
       text="revenue_share",
       color="cluster",
       color_discrete_sequence=CLUSTER_PALETTE
).update_traces(texttemplate="%{text}%",
                textposition="outside"
).update_layout(showlegend=False,
                yaxis=dict(range=[0, 65]),
                xaxis_title="Cluster",
                yaxis_title="Revenue Share (%)"
).show()

In [94]:
px.bar(cluster_revenue,
       x="cluster",
       y="avg_order_value",
       title="Average Order Value by Cluster",
       text="avg_order_value",
       color="cluster",
       color_discrete_sequence=px.colors.qualitative.Set2
).update_traces(texttemplate="%{text:,.0f}",
                textposition="outside"
).update_layout(showlegend=False,
                xaxis_title="Cluster",
                yaxis_title="Average Order Value"
).show()

#### Key Differentiators — Spending and Quantity




In [95]:
px.box(df.assign(cluster=df["cluster"].astype(str)),
       x="cluster",
       y="total_amount",
       color="cluster",
       log_y=True,
       title="Order Value by Cluster (Log Scale)",
       color_discrete_sequence=CLUSTER_PALETTE
).update_layout(showlegend=False,
                xaxis_title="Cluster",
                yaxis_title="Order Value (log scale)"
).show()

In [96]:
px.box(df.assign(cluster=df["cluster"].astype(str)),
       x="cluster",
       y="qty",
       color="cluster",
       title="Purchase Quantity by Cluster",
       color_discrete_sequence=CLUSTER_PALETTE
).update_layout(showlegend=False,
                xaxis_title="Cluster",
                yaxis_title="Quantity"
).show()

#### PCA Projection

PCA reduces the high-dimensional feature space to 2 components for visualization.
Each point is a customer order, colored by cluster assignment. Distance in this
2D projection is approximate — it reflects overall feature similarity, not any
single variable.

In [97]:
from sklearn.decomposition import PCA

X_scaled = preprocessor.fit_transform(df_cluster)
X_pca    = PCA(n_components=2).fit_transform(X_scaled)

pca_df = pd.DataFrame({
    "PC1":     X_pca[:, 0],
    "PC2":     X_pca[:, 1],
    "Cluster": df["cluster"].astype(str)
})

px.scatter(pca_df,
           x="PC1",
           y="PC2",
           color="Cluster",
           title="Customer Segments — PCA Projection",
           color_discrete_sequence=CLUSTER_PALETTE,
           opacity=0.6
).update_layout(xaxis_title="Principal Component 1",
                yaxis_title="Principal Component 2"
).show()

### Cluster Interpretation & Business Personas

| | Cluster 0 | Cluster 1 | Cluster 2 |
|---|---|---|---|
| **Label** | High-Value Buyers | Low-Value Food Buyers | Occasional Buyers |
| **Share of customers** | 24.2% | 23.1% | 52.8% |
| **Share of revenue** | 59.9% | 13.9% | 26.2% |
| **Avg. order value** | ~2,439 | ~590 | ~488 |
| **Avg. qty/order** | ~4.2 | ~1.8 | ~1.5 |
| **Avg. rating** | 4.10 | 2.29 | 4.52 |
| **Key category** | Electronics | Food | Books |
| **Priority** | Retain & upsell | Investigate & recover | Re-engage & upgrade |

---

#### Cluster 0 — High-Value, Electronics-Driven Buyers (24.2% of customers | 59.9% of revenue)

Customers in this cluster generate the highest average order value (~2,439) and
purchase around 4 items per order, driven predominantly by Electronics. Despite
representing less than a quarter of the customer base, they account for nearly
60% of total platform revenue — making them by far the most commercially critical
segment. Customer satisfaction is high (avg rating: 4.10), indicating a healthy
and retainable segment.

Session behavior and demographics are broadly similar to other clusters, confirming
that spending is driven by product choice and purchase intent rather than elevated
engagement.

**Strategy:** Protect and grow this segment through loyalty rewards, early access
to new product launches, extended warranties, financing options, and premium
post-purchase support. Reducing friction in high-value checkout flows is a priority.

---

#### Cluster 1 — Low-Value, Food-Driven Buyers (23.1% of customers | 13.9% of revenue)

This cluster is characterized by low average order value (\~590) and low quantity
per order (\~1.8 items), driven primarily by Food purchases. Despite representing
a similar share of customers as Cluster 0, it generates less than a quarter of
Cluster 0's revenue — the weakest contribution of all three segments.

The most striking signal in this cluster is its average customer rating of 2.29 —
significantly below the platform average of 3.9 and the lowest across all segments.
This satisfaction gap is the defining characteristic of Cluster 1 and should be
the primary focus before any growth investment is directed here.

**Strategy:** Investigate the root cause of low satisfaction scores before
deploying growth campaigns. Once the satisfaction issue is addressed, cross-category
recommendations and bundle offers can be used to increase basket value and migrate
these customers toward higher-margin categories.

---

#### Cluster 2 — Low-Spend, Occasional Buyers (52.8% of customers | 26.2% of revenue)

The largest segment by far, these customers purchase infrequently, buy few items
(~1.5 per order), and gravitate toward low-price categories such as Books. Despite
their size, they generate only 26.2% of revenue — but customer satisfaction is
the highest across all clusters (avg rating: 4.52), suggesting a receptive
audience for re-engagement.

**Strategy:** Re-engagement campaigns, personalized product discovery, and targeted
promotions tied to higher-margin categories are the key levers. The high satisfaction
score means these customers are well-disposed toward the platform — the challenge
is increasing purchase frequency and basket size, not recovering trust.

## 4.2 Drivers of Revenue — Random Forest Regression

### Business Objective
To identify which features most strongly predict order value using supervised
machine learning.

Feature selection note:
- `unit_price` is excluded — it directly constructs the target
  (`total_amount = unit_price x qty - discount_amount`)
- `discount_amount` is excluded for the same reason
- `qty` is retained as a genuine behavioral signal, but its importance score
  should be interpreted with the above relationship in mind

### 4.2.1 Feature Definition

In [98]:
target = "total_amount"

features_num = ["session_min",
                "pag_view",
                "qty",
                "age",
                "delivery_time_days",
                "rating"]

features_cat = ["device",
                "product_cat",
                "pay_method",
                "gender",
                "city"]

features = features_num + features_cat

X = df[features]
y = df[target]

### 4.2.2 Preprocessing & Model Pipeline

In [99]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

preprocessor_rf = ColumnTransformer(transformers=[
    ("num", StandardScaler(), features_num),
    ("cat", OneHotEncoder(drop="first", sparse_output=False), features_cat)
])

# max_depth=10 limits tree complexity to reduce overfitting
# n_estimators=300 provides stable importance estimates
pipeline_rf = Pipeline(steps=[
    ("preprocess", preprocessor_rf),
    ("model",      RandomForestRegressor(n_estimators=300,
                                         max_depth=10,
                                         random_state=0))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)

pipeline_rf.fit(X_train, y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['session_min', 'pag_view',
                                                   'qty', 'age',
                                                   'delivery_time_days',
                                                   'rating']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first',
                                                                sparse_output=False),
                                                  ['device', 'product_cat',
                                                   'pay_method', 'gender',
                                                   'city'])])),
                ('model',
                 RandomForestRegressor(max_depth=10, n_estimators=300,
                                       random_state=0))])

### 4.2.3 Model Evaluation

In [100]:
from sklearn.metrics import r2_score, mean_squared_error

preds = pipeline_rf.predict(X_test)

r2   = r2_score(y_test, preds)
rmse = mean_squared_error(y_test, preds) ** 0.5

print(f"R²:   {r2:.4f}")
print(f"RMSE: {rmse:.2f}")

R²:   0.5882
RMSE: 1295.37


In [101]:
eval_df = pd.DataFrame({
    "Actual":    y_test.values,
    "Predicted": preds
})

max_val = max(eval_df["Actual"].max(), eval_df["Predicted"].max())

px.scatter(eval_df,
           x="Actual",
           y="Predicted",
           opacity=0.4,
           title="Predicted vs Actual Order Value",
           color_discrete_sequence=[PRIMARY_COLOR]
).add_shape(type="line",
            x0=0, y0=0,
            x1=max_val, y1=max_val,
            line=dict(color="red", dash="dash")
).update_layout(xaxis_title="Actual Order Value",
                yaxis_title="Predicted Order Value"
).show()

In [102]:
eval_df["Residual"] = eval_df["Actual"] - eval_df["Predicted"]

px.scatter(eval_df,
           x="Predicted",
           y="Residual",
           opacity=0.4,
           title="Residuals vs Predicted Values",
           color_discrete_sequence=[PRIMARY_COLOR]
).add_hline(y=0,
            line_dash="dash",
            line_color="red"
).update_layout(xaxis_title="Predicted Order Value",
                yaxis_title="Residual"
).show()

### 4.2.4 Feature Importance

In [103]:
feature_names = (
    features_num +
    list(pipeline_rf
         .named_steps["preprocess"]
         .named_transformers_["cat"]
         .get_feature_names_out(features_cat))
)

feat_imp = (pd.Series(pipeline_rf.named_steps["model"].feature_importances_,
                      index=feature_names)
              .sort_values(ascending=False)
              .head(15)
              .reset_index())

feat_imp.columns = ["Feature", "Importance"]

px.bar(feat_imp,
       x="Importance",
       y="Feature",
       orientation="h",
       title="Top 15 Feature Importances — Random Forest",
       text="Importance",
       color="Importance",
       color_continuous_scale="Blues"
).update_traces(texttemplate="%{text:.3f}",
                textposition="outside"
).update_layout(yaxis=dict(autorange="reversed"),
                coloraxis_showscale=False,
                xaxis_title="Importance Score",
                yaxis_title=""
).show()

### Key Insights

- **Product category — specifically Electronics** — is the single strongest
  predictor of order value (importance: 0.316), confirming that what a customer buys matters more than how they behave on the platform. This is consistent with the barbell revenue structure identified in Section 3.4.

- **Quantity** is the second most important feature (0.259). While this partly reflects the mathematical link between qty and total_amount, it also captures a genuine behavioral signal: customers who buy more items spend more, regardless of category.

- **Age, session duration, and page views** contribute modestly and at similar levels (0.062, 0.061, 0.052), suggesting that engagement and demographics play a real but secondary role in predicting spend.

- **Demographic and payment variables** (gender, city, payment method, device) collectively contribute negligible predictive power, reinforcing that revenue is driven by product and quantity decisions rather than customer profile.

- The model achieves an R² of 0.59 and RMSE of 1,295 — moderate predictive power that is reasonable given that key transactional variables were intentionally excluded to avoid leakage.

- The **predicted vs actual plot** shows good fit at low order values but systematic underprediction at high values — a known limitation of Random Forest models, which compress extreme predictions toward the mean.

- The **residuals plot** shows a fan-shaped pattern (heteroscedasticity): errors are small and well-centered for low predicted values but grow at higher values, confirming that model uncertainty increases for high-value orders.

**Business Implication:**  
Revenue is primarily driven by product category and purchase quantity — factors
directly influenced by product portfolio strategy, merchandising, and bundling
decisions. The model's weaker performance on high-value orders suggests electronics purchases involve complex decisions difficult to capture from behavioral data alone.
Future modeling efforts could benefit from separate models per category tier, or the inclusion of additional product-level features such as brand or price tier.

# 5. Strategic Business Recommendations

This section translates the analytical findings from Sections 3 and 4 into
prioritized business actions. Every recommendation is traceable to a specific
finding — the source is referenced in the summary table below.

## Summary of Key Findings

| Finding | Source |
|---|---|
| 24.2% of customers (Cluster 0, Electronics-driven) generate 59.9% of revenue | Section 4.1 |
| Electronics is the strongest predictor of order value (importance: 0.316) | Section 4.2 |
| Quantity is the second strongest predictor (importance: 0.259) | Section 4.2 |
| Cluster 2 represents 52.8% of customers but only 26.2% of revenue | Section 4.1 |
| Cluster 1 generates only 13.9% of revenue despite 23.1% customer share | Section 4.1 |
| Cluster 1 has the lowest avg rating (2.29) vs platform average of 3.9 | Section 4.1 |
| Age, session duration, and page views have limited but real predictive power | Section 4.2 |
| Delivery time and customer ratings are uncorrelated | Section 3.3 |
| Gender and demographic variables show negligible predictive power | Section 3.1 / 4.2 |
| Revenue is product-driven, not behavior- or demographic-driven | Section 3 / 4 |

## Priority 1 — Protect and Grow the High-Value Segment (Cluster 0)

**Finding:** Cluster 0 represents only 24.2% of customers but generates 59.9%
of total revenue, with an average order value of ~2,439 driven by Electronics
purchases. Customer satisfaction is strong (avg rating: 4.10), indicating a
healthy and retainable segment.

**Risk:** This revenue concentration is extreme — losing even a small share of
this segment has an outsized negative impact on overall business performance.

**Recommendations:**

- Introduce a **tiered loyalty program** that rewards high-spend customers with
  early access to new Electronics releases, extended warranties, and priority
  customer support. The goal is to increase switching costs and strengthen
  retention before competitors do.

- Develop **Electronics-specific upsell and cross-sell flows** — accessories,
  protection plans, installation services, and complementary devices. Given that
  quantity is the second strongest revenue predictor (0.259), increasing items
  per order within this segment is the highest-leverage growth opportunity.

- Offer **flexible payment options** (installments, BNPL) for high-value
  Electronics orders to reduce purchase friction and improve conversion on
  premium SKUs.

## Priority 2 — Unlock the Revenue Potential of Cluster 2

**Finding:** Cluster 2 is the largest segment at 52.8% of customers but generates
only 26.2% of revenue. These customers buy infrequently, purchase few items
(avg 1.5 per order), and gravitate toward low-price categories such as Books.
Importantly, satisfaction is the highest across all clusters (avg rating: 4.52)
— this is a receptive audience, not a dissatisfied one. This represents the
platform's largest untapped revenue opportunity.

**Recommendations:**

- Run **category migration campaigns** targeting Cluster 2 customers with
  personalized promotions in higher-margin categories — particularly Electronics
  and Home & Garden. Even a partial migration of this segment toward higher-value
  categories would materially shift overall revenue distribution.

- Use **entry-level bundle offers** to increase items per order. Nudging these
  customers from 1.5 to 2–3 items per order through curated bundles or
  "frequently bought together" features could meaningfully improve their revenue
  contribution.

- Implement **re-engagement triggers** for customers showing inactivity signals,
  using personalized incentives tied to browsing history rather than generic
  discount campaigns. The high satisfaction score means these customers are
  likely to respond positively.

## Priority 3 — Investigate and Recover Cluster 1

**Finding:** Cluster 1 customers represent 23.1% of the customer base but
generate only 13.9% of revenue — the weakest contribution of all three segments
despite not being the smallest. Average order value is low (~590) with ~1.8 items
per order, primarily in Food. The defining signal is an average customer rating
of 2.29, well below the platform average of 3.9.

**Recommendations:**

- **Investigate the root cause of low satisfaction** in this segment as the
  immediate priority. Possible drivers include product quality issues in the Food
  category, unmet delivery expectations, or a mismatch between promotional
  promises and actual experience. Without resolving this, growth campaigns risk
  amplifying a broken experience.

- Once satisfaction is stabilized, introduce **cross-category recommendations**
  at checkout, surfacing Electronics accessories or Home & Garden items alongside
  Food purchases to increase basket composition and order value.

- Test **bundle offers** that pair a Food purchase with a complementary
  higher-margin item at a modest discount, lowering the barrier to cross-category
  exploration.

- Consider **subscription or auto-replenishment models** for Food and other
  consumable categories to secure recurring revenue — but only after the
  satisfaction issue is addressed, as a subscription model for a dissatisfied
  segment would accelerate churn rather than reduce it.

## Priority 4 — Product Portfolio Optimization

**Finding:** Product category is the strongest overall predictor of revenue
(Electronics importance: 0.316). The platform has a clear barbell structure —
Electronics, Home & Garden, and Sports drive value, while Food, Books, Beauty,
and Toys are volume-driven at low margins.

**Recommendations:**

- **Invest in the high-value tier**: prioritize inventory depth, supplier relationships, and logistics quality for Electronics, Home & Garden, and Sports. These categories should receive preferential treatment in search ranking, promotional placement, and catalog expansion.

- **Reposition the low-value tier**: focus on volume efficiency — reducing fulfillment costs, enabling subscriptions, and using these categories as acquisition tools that draw customers onto the platform where they can be cross-sold into higher-value purchases.

- **Direct UX investment toward conversion efficiency** rather than engagement expansion. Session duration and page views show limited predictive power (0.061 and 0.052) — improvements to checkout flow and product discovery will deliver higher returns than features designed to increase time-on-site.

## Priority 5 — Redirect Misallocated Investment

The following are areas where the data found no meaningful effect. Investment
here is unlikely to drive revenue or satisfaction improvements based on this
dataset.

- **Delivery time optimization as a satisfaction lever:** ratings are flat across all observed delivery windows. Operational efficiency remains important, but reducing delivery times is not supported as a satisfaction or retention driver by this data.

- **Demographic targeting by age or gender:** both variables show negligible predictive power for order value. Marketing budget should follow the behavioral clusters identified in Section 4.1 rather than broad demographic profiles.

- **Engagement-led growth strategies:** longer sessions and more page views do not translate into meaningfully higher spending. Strategies designed to increase time-on-site are not supported as revenue drivers.

## Recommendation Summary

| Priority | Focus Area | Key Lever | Revenue Impact |
|---|---|---|---|
| 1 | Retain Cluster 0 | Loyalty, upsell, payment flexibility | Protect 59.9% of revenue |
| 2 | Activate Cluster 2 | Category migration, bundling | Largest growth opportunity |
| 3 | Recover Cluster 1 | Satisfaction investigation, cross-sell | Unlock blocked potential |
| 4 | Product portfolio | Invest in high-value categories | Structural revenue improvement |
| 5 | Redirect spend | Deprioritize delivery & demographics | Cost efficiency |